In [63]:
import pandas as pd

# ── Load all three databases ─────────────────────────────
try:
    bcorp = pd.read_csv("bcorp.csv")
    gots = pd.read_csv("gots.csv")
    india = pd.read_csv("indian_certifications.csv")

    # Standardize column names (Fixes the KeyError)
    if 'certification_type' in bcorp.columns:
        bcorp = bcorp.rename(columns={'certification_type': 'cert_type'})

    print("✅ All databases loaded and standardized successfully!\n")
except FileNotFoundError as e:
    print(f"❌ Error: {e}. Please upload the CSV files first.")

# ── Data passport function ───────────────────────────────
def create_data_passport(df, name):
    print(f"--- {name} OVERVIEW ---")
    print(f"Total Rows: {df.shape[0]} | Total Columns: {df.shape[1]}")

    summary = pd.DataFrame({
        'Data Type': df.dtypes,
        'Total Values': df.count(),
        'Missing Values': df.isnull().sum(),
        '% Missing': (df.isnull().sum() / len(df)) * 100,
        'Unique Values': df.nunique()
    })

    summary = summary.sort_values(by='% Missing', ascending=False)

    print(f"\n1. Data Snapshot (First 5 Rows):")
    display(df.head())

    print(f"\n2. Column Statistics:")
    display(summary.style.background_gradient(cmap='Greens', subset=['% Missing'])
                    .format({'% Missing': '{:.1f}%'}))

    print(f"\n3. Certification Breakdown:")
    # Added dropna=False to ensure we see counts for uncertified (None) brands
    display(df.groupby(['certified', 'cert_type'], dropna=False)
              .size()
              .reset_index(name='count')
              .style.background_gradient(cmap='Greens', subset=['count']))

    print("\n" + "="*50 + "\n")

# ── Run passport for all three ───────────────────────────
create_data_passport(bcorp, "B-CORP CERTIFICATION DATABASE")
create_data_passport(gots, "GOTS CERTIFICATION DATABASE")
create_data_passport(india, "INDIA CERTIFICATIONS DATABASE")

✅ All databases loaded and standardized successfully!

--- B-CORP CERTIFICATION DATABASE OVERVIEW ---
Total Rows: 22 | Total Columns: 4

1. Data Snapshot (First 5 Rows):


,brand,certified,cert_type,category
0,Patagonia,True,B-Corp,apparel
1,Eileen Fisher,True,B-Corp,apparel
2,Allbirds,True,B-Corp,footwear
3,Prana,True,B-Corp,apparel
4,Thought Clothing,True,B-Corp,apparel



2. Column Statistics:


,Data Type,Total Values,Missing Values,% Missing,Unique Values
cert_type,object,17,5,22.7%,1
brand,object,22,0,0.0%,22
certified,bool,22,0,0.0%,2
category,object,22,0,0.0%,5



3. Certification Breakdown:


...

,brand,certified,cert_type,category,country,certifying_body
0,Tata Motors,True,BIS EcoMark,automotive,India,Bureau of Indian Standards
1,Wipro,True,BIS EcoMark,electronics,India,Bureau of Indian Standards
2,Godrej,True,BIS EcoMark,household,India,Bureau of Indian Standards
3,Marico,True,BIS EcoMark,personal care,India,Bureau of Indian Standards
4,ITC,True,GreenPro,manufacturing,India,Confederation of Indian Industry



2. Column Statistics:


,Data Type,Total Values,Missing Values,% Missing,Unique Values
certifying_body,object,10,5,33.3%,3
cert_type,object,10,5,33.3%,3
certified,bool,15,0,0.0%,2
brand,object,15,0,0.0%,15
category,object,15,0,0.0%,7
country,object,15,0,0.0%,1



3. Certification Breakdown:


,certified,cert_type,count
0,False,nan,5
1,True,ASCI Green,2
2,True,BIS EcoMark,4
3,True,GreenPro,4


In [64]:
!pip install transformers streamlit pandas pyngrok beautifulsoup4 requests torch -q

In [65]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification",
                      model="cross-encoder/nli-MiniLM2-L6-H768")
print("✅ Model ready")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/nli-MiniLM2-L6-H768
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model ready


In [66]:
import pandas as pd

data = [
    {"brand": "Patagonia", "certified": True, "certification_type": "B-Corp", "category": "apparel"},
    {"brand": "Eileen Fisher", "certified": True, "certification_type": "B-Corp", "category": "apparel"},
    {"brand": "Allbirds", "certified": True, "certification_type": "B-Corp", "category": "footwear"},
    {"brand": "Prana", "certified": True, "certification_type": "B-Corp", "category": "apparel"},
    {"brand": "Thought Clothing", "certified": True, "certification_type": "B-Corp", "category": "apparel"},
    {"brand": "The Body Shop", "certified": True, "certification_type": "B-Corp", "category": "cosmetics"},
    {"brand": "Weleda", "certified": True, "certification_type": "B-Corp", "category": "cosmetics"},
    {"brand": "Natura", "certified": True, "certification_type": "B-Corp", "category": "cosmetics"},
    {"brand": "Dr. Bronner's", "certified": True, "certification_type": "B-Corp", "category": "cosmetics"},
    {"brand": "Seventh Generation", "certified": True, "certification_type": "B-Corp", "category": "household"},
    {"brand": "Ben & Jerry's", "certified": True, "certification_type": "B-Corp", "category": "food"},
    {"brand": "Innocent Drinks", "certified": True, "certification_type": "B-Corp", "category": "food"},
    {"brand": "Tony's Chocolonely", "certified": True, "certification_type": "B-Corp", "category": "food"},
    {"brand": "Oatly", "certified": True, "certification_type": "B-Corp", "category": "food"},
    {"brand": "tentree", "certified": True, "certification_type": "B-Corp", "category": "apparel"},
    {"brand": "Cotopaxi", "certified": True, "certification_type": "B-Corp", "category": "apparel"},
    {"brand": "Finisterre", "certified": True, "certification_type": "B-Corp", "category": "apparel"},
    # Known greenwashers
    {"brand": "H&M", "certified": False, "certification_type": "None", "category": "apparel"},
    {"brand": "Zara", "certified": False, "certification_type": "None", "category": "apparel"},
    {"brand": "Shein", "certified": False, "certification_type": "None", "category": "apparel"},
    {"brand": "Boohoo", "certified": False, "certification_type": "None", "category": "apparel"},
    {"brand": "Fast Retailing", "certified": False, "certification_type": "None", "category": "apparel"},
]

df = pd.DataFrame(data)
df.to_csv("bcorp.csv", index=False)
print(f"✅ bcorp.csv saved with {len(df)} brands")

✅ bcorp.csv saved with 22 brands


In [67]:
%%writefile app.py
import streamlit as st
import pandas as pd
import requests
import re
from bs4 import BeautifulSoup
from transformers import pipeline

# --- Config ---
BUZZWORDS = [
    "eco-friendly", "green", "natural", "sustainable", "conscious",
    "ethical", "clean", "planet-friendly", "earth-conscious", "non-toxic",
    "biodegradable", "organic", "eco-conscious", "responsible", "pure",
    "carbon neutral", "net zero", "zero waste", "environmentally friendly",
    "loved by nature", "free from harmful chemicals", "toxin-free",
    "chemical-free", "nature-inspired", "plant-based", "earth-friendly",
    "safe for the planet", "free from", "no harmful", "gentle on earth",
    "cruelty-free", "vegan", "clean beauty", "green beauty",
    "sustainability", "climate", "fossil-free", "pre-loved",
    "circular", "commitments", "ambition", "targets", "decarbonise"
]

FUTURE_PROMISES = [
    "aim to", "committed to", "by 2030", "by 2040", "by 2050", "goal is",
    "working towards", "our ambition", "roadmap", "in the future", "strive to",
    "mission to", "pledge"
]

EVIDENCE_KEYWORDS = [
    'according to', 'verified by', 'audited by', 'reported by',
    'certified', 'certification', 'gots', 'b-corp', 'oeko-tex',
    'third party', 'third-party', 'accredited', 'iso', 'standard',
    'control union', 'bureau veritas', 'rainforest alliance',
    'bis ecomark', 'ecomark', 'greenpro', 'cii greenpro',
    'asci', 'isi mark', 'score of', 'independently verified'
]

EVIDENCE_PATTERNS = [
    r'\d+%',
    r'certified', r'certification',
    r'gots', r'b-corp', r'oeko-tex',
    r'audited', r'audit',
    r'third[\s\-]?party',
    r'verified', r'accredited',
    r'iso\s*\d+',
    r'control union', r'bureau veritas',
    r'bis ecomark', r'ecomark', r'greenpro',
    r'rainforest alliance', r'score of \d+'
]

# We only strictly verify certifications that we actually have CSV data for.
TRACKED_CERTS = [
    'gots', 'b-corp', 'b corp', 'bis ecomark', 'greenpro'
]

# Updated Labels for the "Appeals Court" logic
LABELS = ["vague marketing fluff", "objective environmental fact"]

# Universal Topic Filter
ENVIRONMENTAL_TOPICS = BUZZWORDS + FUTURE_PROMISES + EVIDENCE_KEYWORDS + [
    "material", "fabric", "cotton", "wool", "polyester", "plastic", "leather",
    "carbon", "emissions", "water", "energy", "waste", "footprint", "impact",
    "supply chain", "factory", "workers", "packaging", "recycled", "recycling",
    "climate", "environment", "ingredients", "sourcing", "agriculture",
    "forestry", "biodiversity", "renewable", "solar", "power", "manufacturing"
]

# --- Loaders ---
@st.cache_resource
def load_model():
    return pipeline("zero-shot-classification",
                    model="cross-encoder/nli-MiniLM2-L6-H768")

@st.cache_data
def load_databases():
    bcorp = pd.read_csv("bcorp.csv")
    gots = pd.read_csv("gots.csv")
    india = pd.read_csv("indian_certifications.csv")

    if 'certification_type' in bcorp.columns:
        bcorp = bcorp.rename(columns={'certification_type': 'cert_type'})

    bcorp['cert_type'] = bcorp['cert_type'].fillna('None')
    gots['cert_type'] = gots['cert_type'].fillna('None')
    india['cert_type'] = india['cert_type'].fillna('None')

    return bcorp, gots, india

# --- Core Logic ---
def is_relevant_claim(sentence):
    """Checks if the sentence is actually discussing the product/environment."""
    lower = sentence.lower()
    return any(topic in lower for topic in ENVIRONMENTAL_TOPICS)

def has_buzzword(sentence):
    lower = sentence.lower()
    found = [w for w in BUZZWORDS if w in lower]
    return len(found) > 0, found

def has_future_promise(sentence):
    lower = sentence.lower()
    found = [w for w in FUTURE_PROMISES if w in lower]
    return len(found) > 0, found

def has_evidence(sentence):
    lower = sentence.lower()
    return any(re.search(p, lower) for p in EVIDENCE_PATTERNS)

def has_unverified_statistic(sentence):
    lower = sentence.lower()
    has_big_number = bool(re.search(r'\d+\s*(million|billion|ton|tons|kg|lbs)', lower))
    has_source = any(word in lower for word in EVIDENCE_KEYWORDS)
    return has_big_number and not has_source

def check_all_brands(text, bcorp, gots, india):
    text_lower = text.lower()
    matches = []
    for db, db_name in [(bcorp, "B-Corp"), (gots, "GOTS"), (india, "India")]:
        for _, row in db.iterrows():
            brand_lower = str(row["brand"]).lower()
            pattern = r'\b' + re.escape(brand_lower) + r'\b'
            if re.search(pattern, text_lower):
                matches.append({
                    "brand": row["brand"],
                    "certified": bool(row["certified"]),
                    "cert_type": str(row["cert_type"]),
                    "database": db_name
                })
    return matches

def verify_cert_claim(sentence, brand_matches):
    lower = sentence.lower()

    # Only act as a strict lie-detector for databases we actually own
    for cert in TRACKED_CERTS:
        if cert in lower:
            # Normalize text (removes dashes) to catch 'b-corp' vs 'b corp'
            normalized_cert = cert.replace("-", "")
            verified = any(
                normalized_cert in str(r["cert_type"]).lower().replace("-", "") and r["certified"]
                for r in brand_matches
            )
            if not verified:
                return False, cert.upper()

    # If they mention a valid cert we DON'T track (like ISO), give them the benefit of the doubt
    return True, None

def classify_sentence(sentence, classifier, brand_matches):
    buzzword_found, flagged_buzzwords = has_buzzword(sentence)
    promise_found, flagged_promises = has_future_promise(sentence)
    evidence_found = has_evidence(sentence)
    unverified_stat_found = has_unverified_statistic(sentence)

    # Check 1: Future Promises
    if promise_found and not evidence_found:
        return {
            "verdict": "Future Promise (Not Evidence)",
            "score": 0.3,
            "flagged": flagged_promises,
            "path": "Future promise detected",
            "reason": "Corporate goals are not verifiable facts about the current product."
        }

    # Check 2: Unverified Statistics
    if unverified_stat_found:
        return {
            "verdict": "Unverified Statistic",
            "score": 0.2,
            "flagged": flagged_buzzwords,
            "path": "Big number + no source",
            "reason": "Specific numbers used without any source, auditor, or certification mentioned."
        }

    # Check 3: Hardcoded Evidence & Fake Certifications
    if evidence_found:
        verified, cert_name = verify_cert_claim(sentence, brand_matches)
        if not verified and cert_name:
            return {
                "verdict": "Fake Certification Claim",
                "score": 0.0,
                "flagged": flagged_buzzwords,
                "path": "Unverified cert claim",
                "reason": f"Claims {cert_name} certification but this brand does not appear in our {cert_name} database as certified."
            }
        else:
            return {
                "verdict": "Backed Claim",
                "score": 0.9,
                "flagged": flagged_buzzwords,
                "path": "Verified evidence keyword",
                "reason": "Contains an explicit certification, auditor, or verified measurement."
            }

    # Check 4: The AI "Appeals Court"
    result = classifier(sentence, LABELS)
    top_label = result["labels"][0]
    confidence = result["scores"][0]

    if top_label == "objective environmental fact" and confidence > 0.55:
        return {
            "verdict": "Evidence-Based",
            "score": round(confidence, 2),
            "flagged": flagged_buzzwords,
            "path": "NLI Classifier",
            "reason": "AI analyzed the context and found objective facts, even if marketing words were used."
        }
    else:
        if buzzword_found:
            return {
                "verdict": "Vague",
                "score": 0.0,
                "flagged": flagged_buzzwords,
                "path": "Buzzword + AI Failed",
                "reason": "Uses sustainability buzzwords but the AI found no objective facts."
            }
        else:
            return {
                "verdict": "Uncertain / PR Speak",
                "score": 0.4,
                "flagged": [],
                "path": "NLI Classifier",
                "reason": "AI classified this as general marketing or corporate PR."
            }

def audit_product(text, classifier, bcorp, gots, india):
    sentences = [s.strip() for s in re.split(r'[.!?]', text) if len(s.strip()) > 10]
    brand_matches = check_all_brands(text, bcorp, gots, india)

    all_results = []
    valid_claims = []

    for sentence in sentences:
        if not is_relevant_claim(sentence):
            all_results.append({
                "verdict": "Ignored (Website Noise)",
                "score": 0.0,
                "flagged": [],
                "path": "Relevance Gate",
                "reason": "Standard website text. Makes no environmental or product claims.",
                "sentence": sentence
            })
            continue

        r = classify_sentence(sentence, classifier, brand_matches)
        r["sentence"] = sentence
        all_results.append(r)
        valid_claims.append(r)

    final_score = round(sum(r["score"] for r in valid_claims) / len(valid_claims), 2) if valid_claims else 0.0

    is_known_uncertified = any(match["certified"] == False for match in brand_matches)
    if is_known_uncertified:
        final_score = round(max(0.0, final_score - 0.40), 2)

    if final_score < 0.4:
        overall = "Greenwashing Likely"
    elif final_score < 0.7:
        overall = "Uncertain"
    else:
        overall = "Legitimate Claims"

    return {
        "sentences": all_results,
        "final_score": final_score,
        "overall": overall,
        "brand_matches": brand_matches,
        "total_valid_claims": len(valid_claims)
    }

def scrape_url(url):
    try:
        res = requests.get(url, timeout=5, headers={"User-Agent": "Mozilla/5.0"})
        soup = BeautifulSoup(res.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()
        return soup.get_text(separator=". ", strip=True)[:3000]
    except:
        return None

# --- UI ---
st.set_page_config(page_title="Green-Truth Auditor", page_icon="🌿")
st.title("🌿 Green-Truth Auditor")
st.caption("Detecting greenwashing in product descriptions using NLP")

classifier = load_model()
bcorp, gots, india = load_databases()

input_type = st.radio("Input type", ["Paste text", "Enter URL"], horizontal=True)

text = ""
if input_type == "Paste text":
    text = st.text_area("Paste product description here", height=160)
else:
    url = st.text_input("Enter product URL")
    if url:
        with st.spinner("Scraping page..."):
            text = scrape_url(url)
        if text:
            st.success("Page scraped")
            with st.expander("See extracted text"):
                st.write(text[:1000])
        else:
            st.error("Could not scrape that URL. Try pasting the text directly.")

if st.button("Run Audit", type="primary") and text:
    with st.spinner("Auditing claims..."):
        report = audit_product(text, classifier, bcorp, gots, india)

    st.markdown("---")

    color = {
        "Greenwashing Likely": "🚨",
        "Uncertain": "⚠️",
        "Legitimate Claims": "✅"
    }

    col1, col2 = st.columns(2)
    with col1:
        st.metric(
            "Overall Verdict",
            f"{color.get(report['overall'], '')} {report['overall']}",
            f"Score: {report['final_score']} (Based on {report['total_valid_claims']} relevant claims)"
        )
    with col2:
        verdicts = [r["verdict"] for r in report["sentences"]]
        st.write("**Claim breakdown**")
        for v in ["Vague", "Fake Certification Claim", "Unverified Statistic",
                  "Future Promise (Not Evidence)", "Uncertain / PR Speak",
                  "Backed Claim", "Evidence-Based", "Ignored (Website Noise)"]:
            count = verdicts.count(v)
            if count:
                st.write(f"- {v}: {count}")

    st.markdown("---")
    st.subheader("Brand Certification Check")
    st.caption("Shown as context only - does not affect the claim score above.")

    if report["brand_matches"]:
        for match in report["brand_matches"]:
            if match["certified"]:
                st.success(
                    f"✅ **{match['brand']}** is {match['cert_type']} certified "
                    f"(found in {match['database']} database)"
                )
            else:
                st.error(
                    f"❌ **{match['brand']}** is NOT certified "
                    f"(found in {match['database']} database as uncertified). "
                    f"Treat all sustainability claims with high suspicion."
                )
    else:
        st.warning(
            "⚠️ Brand not found in any of our certification databases. "
            "This does not mean they are not certified - "
            "they may just not be in our list."
        )

    st.markdown("---")
    st.subheader("Sentence-by-Sentence Breakdown")

    verdict_emoji = {
        "Vague": "❌",
        "Fake Certification Claim": "🚨",
        "Unverified Statistic": "📉",
        "Future Promise (Not Evidence)": "🗓️",
        "Uncertain / PR Speak": "⚠️",
        "Backed Claim": "✅",
        "Evidence-Based": "📊",
        "Ignored (Website Noise)": "⚪"
    }

    for r in report["sentences"]:
        emoji = verdict_emoji.get(r["verdict"], "ℹ️")
        score_display = "" if r["verdict"] == "Ignored (Website Noise)" else f" | Score: {r['score']}"

        with st.expander(f"{emoji} {r['verdict']}{score_display}"):
            st.write(f"**Sentence:** {r['sentence']}")
            st.write(f"**Why:** {r['reason']}")
            st.write(f"**Path taken:** {r['path']}")
            if r["flagged"]:
                st.write(f"**Flags:** {', '.join(r['flagged'])}")

Overwriting app.py


In [68]:
from pyngrok import ngrok
import subprocess, threading, time

ngrok.kill()
ngrok.set_auth_token("3BYk73p81A7sMteeuksKUQ8jyP9_6TZsRrc6u7QNs4h3KfL2S")  # ← paste it here

thread = threading.Thread(
    target=lambda: subprocess.run(
        ["streamlit", "run", "app.py", "--server.port", "8501"]
    )
)
thread.daemon = True
thread.start()

time.sleep(4)
public_url = ngrok.connect(8501)
print(f"🚀 Live at: {public_url}")

🚀 Live at: NgrokTunnel: "https://alline-lashless-catherina.ngrok-free.dev" -> "http://localhost:8501"


In [69]:
import pandas as pd

test_data = pd.DataFrame({
    "text": [
        "This product is eco-friendly and natural",
        "Certified organic cotton with GOTS certification",
        "We aim to be carbon neutral by 2030",
        "This fabric is ISO 14001 certified",
        "Made with sustainable materials",
        "Audited by Bureau Veritas for emissions",
        "100% recycled plastic bottle",
        "Our goal is to reduce emissions by 2040"
    ],
    "label": [0, 1, 0, 1, 0, 1, 1, 0]
})

test_data.to_csv("test_data.csv", index=False)
print("✅ Test dataset created")

✅ Test dataset created


In [70]:
test_df = pd.read_csv("test_data.csv")
test_df.head()


,text,label
0,This product is eco-friendly and natural,0
1,Certified organic cotton with GOTS certification,1
2,We aim to be carbon neutral by 2030,0
3,This fabric is ISO 14001 certified,1
4,Made with sustainable materials,0


In [71]:
def predict_label(text):
    result = classifier(text, LABELS)
    top_label = result["labels"][0]

    if top_label == "specific and verifiable environmental evidence":
        return 1
    else:
        return 0

In [72]:
LABELS = ["vague marketing or future promises", "specific and verifiable environmental evidence"]

def predict_label(text):
    # classifier is assumed to be globally available from cell O-iUB7M0uKhJ
    result = classifier(text, LABELS)
    top_label = result["labels"][0]

    if top_label == "specific and verifiable environmental evidence":
        return 1
    else:
        return 0

test_df["predicted"] = test_df["text"].apply(predict_label)
test_df

,text,label,predicted
0,This product is eco-friendly and natural,0,1
1,Certified organic cotton with GOTS certification,1,1
2,We aim to be carbon neutral by 2030,0,1
3,This fabric is ISO 14001 certified,1,1
4,Made with sustainable materials,0,1
5,Audited by Bureau Veritas for emissions,1,1
6,100% recycled plastic bottle,1,1
7,Our goal is to reduce emissions by 2040,0,1


In [73]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(test_df["label"], test_df["predicted"])
precision = precision_score(test_df["label"], test_df["predicted"])
recall = recall_score(test_df["label"], test_df["predicted"])
f1 = f1_score(test_df["label"], test_df["predicted"])

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.5
Precision: 0.5
Recall: 1.0
F1 Score: 0.6666666666666666


In [74]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(test_df["label"], test_df["predicted"])
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[0 4]
 [0 4]]


In [75]:
def full_system_predict(text):
    report = audit_product(text, classifier, bcorp, gots, india)

    if report["overall"] == "Legitimate Claims":
        return 1
    else:
        return 0

In [76]:
import re
import pandas as pd

# Re-define global variables from app.py that are needed
BUZZWORDS = [
    "eco-friendly", "green", "natural", "sustainable", "conscious",
    "ethical", "clean", "planet-friendly", "earth-conscious", "non-toxic",
    "biodegradable", "organic", "eco-conscious", "responsible", "pure",
    "carbon neutral", "net zero", "zero waste", "environmentally friendly",
    "loved by nature", "free from harmful chemicals", "toxin-free",
    "chemical-free", "nature-inspired", "plant-based", "earth-friendly",
    "safe for the planet", "free from", "no harmful", "gentle on earth",
    "cruelty-free", "vegan", "clean beauty", "green beauty",
    "sustainability", "climate", "fossil-free", "pre-loved",
    "circular", "commitments", "ambition", "targets", "decarbonise"
]

FUTURE_PROMISES = [
    "aim to", "committed to", "by 2030", "by 2040", "by 2050", "goal is",
    "working towards", "our ambition", "roadmap", "in the future", "strive to",
    "mission to", "pledge"
]

EVIDENCE_KEYWORDS = [
    'according to', 'verified by', 'audited by', 'reported by',
    'certified', 'certification', 'gots', 'b-corp', 'oeko-tex',
    'third party', 'third-party', 'accredited', 'iso', 'standard',
    'control union', 'bureau veritas', 'rainforest alliance',
    'bis ecomark', 'ecomark', 'greenpro', 'cii greenpro',
    'asci', 'isi mark', 'score of', 'independently verified'
]

EVIDENCE_PATTERNS = [
    r'\d+%',
    r'certified', r'certification',
    r'gots', r'b-corp', r'oeko-tex',
    r'audited', r'audit',
    r'third[\s\-]?party',
    r'verified', r'accredited',
    r'iso\s*\d+',
    r'control union', r'bureau veritas',
    r'bis ecomark', r'ecomark', r'greenpro',
    r'rainforest alliance', r'score of \d+'
]

CERT_NAMES = [
    'gots', 'b-corp', 'oeko-tex', 'rainforest alliance',
    'fair trade', 'control union', 'bureau veritas',
    'bis ecomark', 'ecomark', 'greenpro', 'cii greenpro',
    'asci', 'isi mark'
]

# LABELS is already defined in a previous cell (JBLU7-MxT3g3) and is globally available.

# NEW: Universal Topic Filter
ENVIRONMENTAL_TOPICS = BUZZWORDS + FUTURE_PROMISES + EVIDENCE_KEYWORDS + [
    "material", "fabric", "cotton", "wool", "polyester", "plastic", "leather",
    "carbon", "emissions", "water", "energy", "waste", "footprint", "impact",
    "supply chain", "factory", "workers", "packaging", "recycled", "recycling",
    "climate", "environment", "ingredients", "sourcing", "agriculture",
    "forestry", "biodiversity", "renewable", "solar", "power", "manufacturing"
]

# --- Core Logic Functions from app.py ---
def is_relevant_claim(sentence):
    """Checks if the sentence is actually discussing the product/environment."""
    lower = sentence.lower()
    return any(topic in lower for topic in ENVIRONMENTAL_TOPICS)

def has_buzzword(sentence):
    lower = sentence.lower()
    found = [w for w in BUZZWORDS if w in lower]
    return len(found) > 0, found

def has_future_promise(sentence):
    lower = sentence.lower()
    found = [w for w in FUTURE_PROMISES if w in lower]
    return len(found) > 0, found

def has_evidence(sentence):
    lower = sentence.lower()
    return any(re.search(p, lower) for p in EVIDENCE_PATTERNS)

def has_unverified_statistic(sentence):
    lower = sentence.lower()
    has_big_number = bool(re.search(r'\d+\s*(million|billion|ton|tons|kg|lbs)', lower))
    has_source = any(word in lower for word in EVIDENCE_KEYWORDS)
    return has_big_number and not has_source

def check_all_brands(text, bcorp, gots, india):
    text_lower = text.lower()
    matches = []
    for db, db_name in [(bcorp, "B-Corp"), (gots, "GOTS"), (india, "India")]:
        for _, row in db.iterrows():
            brand_lower = str(row["brand"]).lower()
            pattern = r'\b' + re.escape(brand_lower) + r'\b'
            if re.search(pattern, text_lower):
                matches.append({
                    "brand": row["brand"],
                    "certified": bool(row["certified"]),
                    "cert_type": str(row["cert_type"]),
                    "database": db_name
                })
    return matches

def verify_cert_claim(sentence, brand_matches):
    lower = sentence.lower()
    for cert in CERT_NAMES:
        if cert in lower:
            verified = any(
                cert.lower() in str(r["cert_type"]).lower() and r["certified"]
                for r in brand_matches
            )
            if not verified:
                return False, cert.upper()
    return True, None

def classify_sentence(sentence, classifier, brand_matches):
    buzzword_found, flagged_buzzwords = has_buzzword(sentence)
    promise_found, flagged_promises = has_future_promise(sentence)

    if promise_found and not has_evidence(sentence):
        return {
            "verdict": "Future Promise (Not Evidence)",
            "score": 0.3,
            "flagged": flagged_promises,
            "path": "future promise detected",
            "reason": "Corporate goals and future pledges are not verifiable facts about the current product."
        }

    if buzzword_found:
        if has_evidence(sentence):
            if has_unverified_statistic(sentence):
                return {
                    "verdict": "Unverified Statistic",
                    "score": 0.2,
                    "flagged": flagged_buzzwords,
                    "path": "buzzword + big number, no source",
                    "reason": "Contains specific numbers but no certifying body or source to back them up."
                }
            verified, cert_name = verify_cert_claim(sentence, brand_matches)
            if not verified:
                return {
                    "verdict": "Fake Certification Claim",
                    "score": 0.0,
                    "flagged": flagged_buzzwords,
                    "path": "buzzword + unverified cert claim",
                    "reason": f"Claims {cert_name} certification but this brand does not appear in our {cert_name} database as certified."
                }
            else:
                return {
                    "verdict": "Backed Claim",
                    "score": 0.9,
                    "flagged": flagged_buzzwords,
                    "path": "buzzword + verified evidence",
                    "reason": "Contains buzzword but also references a certification or verified evidence."
                }
        if has_unverified_statistic(sentence):
            return {
                "verdict": "Unverified Statistic",
                "score": 0.2,
                "flagged": flagged_buzzwords,
                "path": "buzzword + big number, no source",
                "reason": "Specific numbers used without any source, auditor, or certification mentioned."
            }
        return {
            "verdict": "Vague",
            "score": 0.0,
            "flagged": flagged_buzzwords,
            "path": "buzzword, no evidence",
            "reason": "Uses feel-good sustainability language with no evidence or certification to back it up."
        }
    else:
        if has_unverified_statistic(sentence):
            return {
                "verdict": "Unverified Statistic",
                "score": 0.2,
                "flagged": [],
                "path": "big number, no source",
                "reason": "Specific numbers used without any source, auditor, or certification mentioned."
            }

        result = classifier(sentence, LABELS)
        confidence = result["scores"][0]
        top_label = result["labels"][0]

        if top_label == "specific and verifiable environmental evidence" and confidence > 0.6:
            return {
                "verdict": "Evidence-Based",
                "score": round(confidence, 2),
                "flagged": [],
                "path": "NLI classifier",
                "reason": "No buzzwords detected. AI judged this as containing specific environmental evidence."
            }
        else:
            return {
                "verdict": "Uncertain / PR Speak",
                "score": 0.4,
                "flagged": [],
                "path": "NLI classifier",
                "reason": "AI classified this as vague marketing, corporate PR, or could not find specific evidence."
            }

def audit_product(text, classifier, bcorp, gots, india):
    sentences = [s.strip() for s in re.split(r'[.!?]', text) if len(s.strip()) > 10]
    brand_matches = check_all_brands(text, bcorp, gots, india)

    all_results = []
    valid_claims = []

    for sentence in sentences:
        if not is_relevant_claim(sentence):
            all_results.append({
                "verdict": "Ignored (Website Noise)",
                "score": 0.0,
                "flagged": [],
                "path": "Relevance Gate",
                "reason": "Standard website text. Makes no environmental or product claims.",
                "sentence": sentence
            })
            continue

        r = classify_sentence(sentence, classifier, brand_matches)
        r["sentence"] = sentence
        all_results.append(r)
        valid_claims.append(r)

    final_score = round(sum(r["score"] for r in valid_claims) / len(valid_claims), 2) if valid_claims else 0.0

    is_known_uncertified = any(match["certified"] == False for match in brand_matches)
    if is_known_uncertified:
        final_score = round(max(0.0, final_score - 0.40), 2)

    if final_score < 0.4:
        overall = "Greenwashing Likely"
    elif final_score < 0.7:
        overall = "Uncertain"
    else:
        overall = "Legitimate Claims"

    return {
        "sentences": all_results,
        "final_score": final_score,
        "overall": overall,
        "brand_matches": brand_matches,
        "total_valid_claims": len(valid_claims)
    }

test_df["system_pred"] = test_df["text"].apply(full_system_predict)

accuracy = accuracy_score(test_df["label"], test_df["system_pred"])
print("System Accuracy:", accuracy)


System Accuracy: 0.875


In [77]:
def map_verdict_to_label(verdict):
    if verdict in ["Evidence-Based", "Backed Claim"]:
        return 1   # Genuine
    else:
        return 0   # Greenwashing / unclear